### Setup

In [1]:
import mlflow
from openai import OpenAI
import time
import logging

### Secure API Key Loading

In [2]:
with open("../api_key.txt", "r") as f:
    API_KEY = f.read().strip()

client = OpenAI(api_key=API_KEY)


### Configure MLflow Experiment

In [3]:
logging.getLogger("mlflow").setLevel(logging.ERROR)
mlflow.set_experiment("llm_experiment_tracking_demo")

2026/01/20 12:38:04 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/20 12:38:04 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/20 12:38:04 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/20 12:38:04 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/home/khan/PS/LLMOps_Evaluation_Observability_Quality/mlruns/1', creation_time=1768236197964, experiment_id='1', last_update_time=1768236197964, lifecycle_stage='active', name='llm_experiment_tracking_demo', tags={}>

### Simple LLM Evaluation Function

In [4]:
def run_llm_experiment(model_name, temperature, prompt):
    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )

    latency = time.time() - start_time
    output = response.choices[0].message.content

    # Simple proxy metrics
    output_length = len(output)
    success = int(len(output.strip()) > 0)

    return output, latency, output_length, success

### Log Two Experiments (Different Configs)

In [5]:
# Run 1 — Low Temperature
with mlflow.start_run(run_name="low_temperature_run"):
    mlflow.log_param("model", "gpt-4o-mini")
    mlflow.log_param("temperature", 0.0)

    output, latency, length, success = run_llm_experiment(
        model_name="gpt-4o-mini",
        temperature=0.0,
        prompt="Explain agentic AI in one sentence."
    )

    mlflow.log_metric("latency", latency)
    mlflow.log_metric("output_length", length)
    mlflow.log_metric("success", success)

    print("Run 1 Output:\n", output)


Run 1 Output:
 Agentic AI refers to artificial intelligence systems that possess the ability to act autonomously, make decisions, and pursue goals in a manner that resembles human agency.


In [6]:
# Run 2 — Higher Temperature
with mlflow.start_run(run_name="higher_temperature_run"):
    mlflow.log_param("model", "gpt-4o-mini")
    mlflow.log_param("temperature", 0.7)

    output, latency, length, success = run_llm_experiment(
        model_name="gpt-4o-mini",
        temperature=0.7,
        prompt="Explain agentic AI in one sentence."
    )

    mlflow.log_metric("latency", latency)
    mlflow.log_metric("output_length", length)
    mlflow.log_metric("success", success)

    print("Run 2 Output:\n", output)


Run 2 Output:
 Agentic AI refers to artificial intelligence systems that possess the ability to act autonomously, make decisions, and pursue goals independently, often exhibiting behaviors akin to agency.


### Compare Experiments Side by Side (MLflow UI)

In [7]:
# Run the following and access MLflow UI on http://localhost:5000
!mlflow ui

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
2026/01/20 12:40:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/20 12:40:08 INFO mlflow.store.db.utils: Updating database tables
2026/01/20 12:40:08 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/20 12:40:08 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/20 12:40:09 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/20 12:40:09 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/20 12:40:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/20 12:40:09 INFO mlflow.store.db.utils: Updating database tables
2026/01/20 12:40:09 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/20 12:40:09 INFO alembic.runtime.migration: Will assume non-transactional DDL.
[MLflow] Security middleware enabled with default settings (loca

### Identify & Restore the “Best” Run

In [8]:
runs = mlflow.search_runs(
    experiment_names=["llm_experiment_tracking_demo"]
)

best_run = runs.sort_values(
    by=["metrics.success", "metrics.latency"],
    ascending=[False, True]
).iloc[0]

print("Best Run ID:", best_run.run_id)
print("Best Run Params:", best_run.filter(like="params."))

Best Run ID: f030677e8a0c483bb8638cba1a21710b
Best Run Params: params.model          gpt-4o-mini
params.temperature            0.7
Name: 0, dtype: object
